In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score


In [4]:
df = pd.read_csv("taxi_trip_pricing.csv")
df.head()


,Trip_Distance_km,Time_of_Day,Day_of_Week,Passenger_Count,Traffic_Conditions,Weather,Base_Fare,Per_Km_Rate,Per_Minute_Rate,Trip_Duration_Minutes,Trip_Price
0,19.35,Morning,Weekday,3.0,Low,Clear,3.56,0.80,0.32,53.82,36.2624
1,47.59,Afternoon,Weekday,1.0,High,Clear,NaN,0.62,0.43,40.57,NaN
2,36.87,Evening,Weekend,1.0,High,Clear,2.70,1.21,0.15,37.27,52.9032
3,30.33,Evening,Weekday,4.0,Low,NaN,3.48,0.51,0.15,116.81,36.4698
4,NaN,Evening,Weekday,3.0,High,Clear,2.93,0.63,0.32,22.64,15.6180


In [5]:
for col in df.select_dtypes(include=np.number).columns:
    df[col].fillna(df[col].median(), inplace=True)

for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)


C:\Users\Local User\AppData\Local\Temp\ipykernel_20380\3377565077.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\Local User\AppData\Local\Temp\ipykernel_20380\3377565077.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.


In [6]:
df.isnull().sum()

Trip_Distance_km         0
Time_of_Day              0
Day_of_Week              0
Passenger_Count          0
Traffic_Conditions       0
Weather                  0
Base_Fare                0
Per_Km_Rate              0
Per_Minute_Rate          0
Trip_Duration_Minutes    0
Trip_Price               0
dtype: int64

In [7]:
target = "Trip_Price"

X = df.drop(columns=[target])
y = df[target]


In [8]:
le = LabelEncoder()


for col in X.select_dtypes(include='object').columns:
    X[col] = le.fit_transform(X[col])
 


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [10]:
dt = DecisionTreeRegressor(random_state=42)
rf = RandomForestRegressor(n_estimators=200, random_state=42)
svr = SVR()


In [11]:
dt.fit(X_train, y_train)
rf.fit(X_train, y_train)
svr.fit(X_train, y_train)


,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,1.0
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [12]:
dt_pred = dt.predict(X_test)
rf_pred = rf.predict(X_test)
svr_pred = svr.predict(X_test)


In [13]:
print("Score of DT on training set:",dt.score(X_train,y_train))
print("Score of DT on test set:",dt.score(X_test,y_test))

Score of DT on training set: 1.0
Score of DT on test set: 0.808500653308738


In [14]:
print("Score of  on training set:",rf.score(X_train,y_train))
print("Score of  on test set:",rf.score(X_test,y_test))

Score of  on training set: 0.9894203784192044
Score of  on test set: 0.9317187539790992


In [15]:
print("Score of  on training set:",svr.score(X_train,y_train))
print("Score of  on test set:",svr.score(X_test,y_test))

Score of  on training set: 0.30098129821425945
Score of  on test set: 0.44223662162865673


In [16]:
print("Decision Tree")
print("MAE:", mean_absolute_error(y_test, dt_pred))
print("R2 :", r2_score(y_test, dt_pred))

print("\nRandom Forest")
print("MAE:", mean_absolute_error(y_test, rf_pred))
print("R2 :", r2_score(y_test, rf_pred))

print("\nSVR")
print("MAE:", mean_absolute_error(y_test, svr_pred))
print("R2 :", r2_score(y_test, svr_pred))


Decision Tree
MAE: 9.298761397353685
R2 : 0.808500653308738

Random Forest
MAE: 5.365734343957042
R2 : 0.9317187539790992

SVR
MAE: 14.498486089155929
R2 : 0.44223662162865673


In [17]:
results = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest", "SVR"],
    "MAE": [
        mean_absolute_error(y_test, dt_pred),
        mean_absolute_error(y_test, rf_pred),
        mean_absolute_error(y_test, svr_pred)
    ],
    "R2 Score": [
        r2_score(y_test, dt_pred),
        r2_score(y_test, rf_pred),
        r2_score(y_test, svr_pred)
    ]
})

results


,Model,MAE,R2 Score
0,Decision Tree,9.298761,0.808501
1,Random Forest,5.365734,0.931719
2,SVR,14.498486,0.442237


In [18]:
importance = pd.Series(rf.feature_importances_, index=X.columns)
importance.sort_values(ascending=False)


Trip_Distance_km         0.818895
Per_Km_Rate              0.073002
Trip_Duration_Minutes    0.055426
Per_Minute_Rate          0.029910
Base_Fare                0.011711
Passenger_Count          0.003291
Traffic_Conditions       0.002806
Time_of_Day              0.002261
Weather                  0.001388
Day_of_Week              0.001310
dtype: float64

In [ ]:
new_trip = pd.DataFrame({
    "Trip_Distance_km": [19.35],
    "Time_of_Day": ["Morning"],
    "Day_of_Week": ["Weekday"],
    "Passenger_Count": [3.0],
    "Traffic_Conditions": ["Low"],
    "Weather": ["Clear"],
    "Base_Fare": [3.56],
    "Per_Km_Rate": [0.80],
    "Per_Minute_Rate": [0.32],
    "Trip_Duration_Minutes": [53.82]
})




In [20]:
for col in new_trip.select_dtypes(include='object').columns:
    new_trip[col] = le.fit_transform(new_trip[col])


In [21]:
predicted_price = rf.predict(new_trip)
print("Predicted Trip Price:", predicted_price[0])


Predicted Trip Price: 35.212238999999926


In [22]:
import pickle

In [ ]:
with open("Taxi.pkl","wb")as f:
    pickle.dump(rf,f)

NameError: name 'lr' is not defined